In [1]:
# 1. import libraries

import pandas as pd
import numpy as np
import os

In [2]:
# 2. load the clean tnbc data

data_path = r"E:\apply\journal publication\tnbc-diffusion\data\processed"

tnbc = pd.read_csv(os.path.join(data_path, "tnbc_patients.csv"))

print("loaded patients:", len(tnbc))
print("columns:", tnbc.columns.tolist())

loaded patients: 132
columns: ['Patient_ID', 'Arm', 'HR', 'HER2', 'MP', 'pCR', 'Age_at_Screening', 'Race', 'menopausal_status', 'ethnicity', 'VOLUME_TUM_BLU_V10', 'VOLUME_TUM_BLU_V20', 'VOLUME_TUM_BLU_V30', 'VOLUME_TUM_BLU_V40', 'SPHERICITY_T0', 'SPHERICITY_T1', 'SPHERICITY_T2', 'SPHERICITY_T3', 'LD_T0', 'LD_T1', 'LD_T2', 'LD_T3', 'BPE_5slice_mean_T0', 'BPE_5slice_mean_T1', 'BPE_5slice_mean_T2', 'BPE_5slice_mean_T3', 'FTV_pch_T0_T1', 'FTV_pch_T0_T2', 'FTV_pch_T0_T3', 'Sphericity_pch_T0_T1', 'Sphericity_pch_T0_T2', 'Sphericity_pch_T0_T3', 'LD_pch_T0_T1', 'LD_pch_T0_T2', 'LD_pch_T0_T3', 'BPE_pch_T0_T1', 'BPE_pch_T0_T2', 'BPE_pch_T0_T3']


In [3]:
# 3. rank patients by tumor size at baseline for image download priority

tnbc_ranked = tnbc[["Patient_ID", "pCR", "LD_T0", "SPHERICITY_T0", "VOLUME_TUM_BLU_V10"]].copy()
tnbc_ranked = tnbc_ranked.sort_values("LD_T0", ascending=False).reset_index(drop=True)

print(tnbc_ranked.head(20))

    Patient_ID  pCR  LD_T0  SPHERICITY_T0  VOLUME_TUM_BLU_V10
0       910706    1   10.8       0.155336           38.019287
1       829491    0   10.4       0.099783          109.172920
2       622315    0   10.2       0.110812          214.371580
3       233191    0   10.1       0.105483           95.102139
4       502486    0    9.9       0.069566          168.715630
5       489504    1    9.1       0.122291           32.252344
6       535779    0    9.0       0.110122           47.847039
7       164468    0    9.0       0.076689          135.920040
8       571995    0    8.0       0.222885           35.725918
9       934906    0    7.9       0.200344           72.601281
10      196024    0    7.8       0.156231          169.605320
11      578975    1    7.8       0.149489           12.983110
12      955035    0    7.6       0.161112           99.213495
13      102212    0    7.4       0.212387           31.107110
14      625854    1    7.3       0.108839          163.865840
15      

In [4]:
# 4. select top 20 patients by tumor size for first image download batch

top20 = tnbc_ranked.head(20)

save_path = r"E:\apply\journal publication\tnbc-diffusion\data\processed"
top20["Patient_ID"].to_csv(os.path.join(save_path, "priority_download_ids.csv"), index=False)

print("top 20 patient ids saved")
print(top20["Patient_ID"].tolist())

top 20 patient ids saved
[910706, 829491, 622315, 233191, 502486, 489504, 535779, 164468, 571995, 934906, 196024, 578975, 955035, 102212, 625854, 618240, 559263, 402384, 115638, 127675]


In [5]:
# 5. check pcr balance in top 20

print("complete response in top 20:", top20["pCR"].sum())
print("no response in top 20:", (top20["pCR"] == 0).sum())

complete response in top 20: 4
no response in top 20: 16


In [6]:
# 6. select a balanced batch of 20 patients, 10 responders and 10 non-responders

responders = tnbc_ranked[tnbc_ranked["pCR"] == 1].head(10)
non_responders = tnbc_ranked[tnbc_ranked["pCR"] == 0].head(10)

balanced_batch = pd.concat([responders, non_responders]).reset_index(drop=True)

print(balanced_batch[["Patient_ID", "pCR", "LD_T0", "SPHERICITY_T0"]])

    Patient_ID  pCR  LD_T0  SPHERICITY_T0
0       910706    1   10.8       0.155336
1       489504    1    9.1       0.122291
2       578975    1    7.8       0.149489
3       625854    1    7.3       0.108839
4       208303    1    5.9       0.194695
5       774840    1    5.7       0.138298
6       727804    1    5.4       0.174513
7       564234    1    5.3       0.162843
8       252748    1    5.0       0.294321
9       697098    1    4.8       0.171698
10      829491    0   10.4       0.099783
11      622315    0   10.2       0.110812
12      233191    0   10.1       0.105483
13      502486    0    9.9       0.069566
14      535779    0    9.0       0.110122
15      164468    0    9.0       0.076689
16      571995    0    8.0       0.222885
17      934906    0    7.9       0.200344
18      196024    0    7.8       0.156231
19      955035    0    7.6       0.161112


In [7]:
# 7. save the balanced batch ids for image download

balanced_batch["Patient_ID"].to_csv(
    os.path.join(save_path, "balanced_download_ids.csv"), index=False
)

print("complete response patients:", balanced_batch["pCR"].sum())
print("no response patients:", (balanced_batch["pCR"] == 0).sum())
print()
print("patient ids to download:")
print(balanced_batch["Patient_ID"].tolist())

complete response patients: 10
no response patients: 10

patient ids to download:
[910706, 489504, 578975, 625854, 208303, 774840, 727804, 564234, 252748, 697098, 829491, 622315, 233191, 502486, 535779, 164468, 571995, 934906, 196024, 955035]
